In [1]:
import sys
sys.path.append("../..")

from pathlib import Path

import pandas as pd
from truckscenes import TruckScenes

from src.config import DATA_ROOT, METADATA_ROOT, SENSOR_ROOT
from src.dataset_filtering import prepare_available_sample_data
from src.sensor_observation import build_sensor_data_df
from src.annotation_velocity import build_vehicle_reference_df
from src.radar_velocity import (
    build_target_radar_points_df,
    build_radar_radial_df,
    build_radar_velocity_df,
    build_radar_radial_validation_df,
)

RESULT_DIR = Path("../../results/condition")

In [2]:
trucksc = TruckScenes(dataroot=DATA_ROOT)

available_sample_data, summary, extra_files = prepare_available_sample_data()
display(summary)

sensor_data_df = build_sensor_data_df(
    metadata_root=METADATA_ROOT,
    sample_data_df=available_sample_data,
)

keyframe_df = sensor_data_df[
    sensor_data_df["Is Key Frame"].eq(True)
].copy()

radar_observation_df = (
    keyframe_df[keyframe_df["Modality"].eq("RADAR")]
    .reset_index(drop=True)
)

print("RADAR observations:", len(radar_observation_df))

Auto-detected latest version: v1.2-mini
Loading truckscenes tables for version v1.2-mini...
11 attribute,
18 calibrated_sensor,
27 category,
20090 ego_motion_cabin,
20089 ego_motion_chassis,
20116 ego_pose,
1094 instance,
400 sample,
25750 sample_annotation,
43556 sample_data,
10 scene,
18 sensor,
4 visibility,
10 weather_annotation,
Done loading in 0.738 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


f:\CITS3200\Boeing-RADAR-Autonomous-\.venv311\Lib\site-packages\truckscenes\truckscenes.py:100: UserWarning: The visualization dependencies are not installed on your system! Run 'pip install "truckscenes-devkit[all]"'.
  warnings.warn('''The visualization dependencies are not installed on your system! '''


Metadata records                    43556
Available local files               43556
Unavailable locally                     0
Extra / unreferenced local files        0
Name: Dataset Filtering Summary, dtype: int64

RADAR observations: 2400


In [3]:
vehicle_reference_df = build_vehicle_reference_df(
    trucksc.sample,
    trucksc.sample_annotation,
    trucksc.instance,
    trucksc.category,
)

print("Reference intervals:", len(vehicle_reference_df))

Reference intervals: 15743


In [4]:
vehicle_reference_df.to_csv(
    RESULT_DIR / "vehicle_reference_df.csv",
    index=False,
)

In [5]:
radar_tokens = set(radar_observation_df["Token"])
radar_sample_tokens = set(radar_observation_df["Sample Token"])

radar_sample_data = available_sample_data[
    available_sample_data["token"].isin(radar_tokens)
].copy()

radar_sample_data_lookup = (
    radar_sample_data.set_index("token").to_dict("index")
)

ego_pose_tokens = set(radar_sample_data["ego_pose_token"].dropna())
radar_ego_pose_lookup = {
    x["token"]: x
    for x in trucksc.ego_pose
    if x["token"] in ego_pose_tokens
}

calibrated_sensor_tokens = set(
    radar_sample_data["calibrated_sensor_token"].dropna()
)
radar_calibrated_sensor_lookup = {
    x["token"]: x
    for x in trucksc.calibrated_sensor
    if x["token"] in calibrated_sensor_tokens
}

radar_annotations = [
    x
    for x in trucksc.sample_annotation
    if x["sample_token"] in radar_sample_tokens
]

In [6]:
target_radar_points_df = build_target_radar_points_df(
    radar_observation_df,
    radar_annotations,
    trucksc.instance,
    trucksc.category,
    radar_sample_data_lookup,
    radar_ego_pose_lookup,
    radar_calibrated_sensor_lookup,
    SENSOR_ROOT,
)

print("Target RADAR points:", len(target_radar_points_df))

Target RADAR points: 52184


In [7]:
radar_radial_df = build_radar_radial_df(
    target_radar_points_df,
    radar_sample_data_lookup,
    radar_ego_pose_lookup,
    radar_calibrated_sensor_lookup,
    trucksc.ego_motion_cabin,
)

print("RADAR radial points:", len(radar_radial_df))

RADAR radial points: 52184


In [8]:
target_radar_points_df = build_target_radar_points_df(
    radar_observation_df,
    radar_annotations,
    trucksc.instance,
    trucksc.category,
    radar_sample_data_lookup,
    radar_ego_pose_lookup,
    radar_calibrated_sensor_lookup,
    SENSOR_ROOT,
)

In [9]:
radar_radial_df.to_csv(
    RESULT_DIR / "radar_radial_df.csv",
    index=False,
)

In [10]:
radar_velocity_df = build_radar_velocity_df(
    radar_radial_df,
    vehicle_reference_df,
)

print("RADAR reconstructable intervals:", len(radar_velocity_df))
print("RADAR stable intervals:", radar_velocity_df["is_stable_2d"].sum())

RADAR reconstructable intervals: 7706
RADAR stable intervals: 3016


In [11]:
radar_velocity_df.to_csv(
    RESULT_DIR / "radar_velocity_df.csv",
    index=False,
)

In [12]:
radar_validation_df = build_radar_radial_validation_df(
    radar_radial_df,
    vehicle_reference_df,
)

print(
    "Validated RADAR radial observations:",
    len(radar_validation_df),
)

radar_validation_df.head()

Validated RADAR radial observations: 50823


,annotation_token,sensor,radar_radial_velocity_mps,los_global_x,los_global_y,reference_radial_velocity_mps,radial_residual_mps,radial_abs_error_mps,start_annotation_token,end_annotation_token,instance_token,category
0,50873dd5195b47bc9544ac2821e054e0,RADAR_LEFT_FRONT,-0.058475,0.380595,0.918101,-0.093246,0.034771,0.034771,50873dd5195b47bc9544ac2821e054e0,9f0e8f997dc647bf9a0a20c679b1804d,163b7c695f7243779ef5e5f0fd844063,vehicle.car
1,50873dd5195b47bc9544ac2821e054e0,RADAR_LEFT_FRONT,-0.055880,0.359044,0.926790,-0.095827,0.039947,0.039947,50873dd5195b47bc9544ac2821e054e0,9f0e8f997dc647bf9a0a20c679b1804d,163b7c695f7243779ef5e5f0fd844063,vehicle.car
2,9f0e8f997dc647bf9a0a20c679b1804d,RADAR_RIGHT_FRONT,-0.029377,0.253124,0.959433,-0.161167,0.131790,0.131790,9f0e8f997dc647bf9a0a20c679b1804d,2c8422f4ac674f949fa1124dcaa1fa5e,163b7c695f7243779ef5e5f0fd844063,vehicle.car
3,9f0e8f997dc647bf9a0a20c679b1804d,RADAR_LEFT_FRONT,-0.039039,0.395074,0.911228,-0.151928,0.112889,0.112889,9f0e8f997dc647bf9a0a20c679b1804d,2c8422f4ac674f949fa1124dcaa1fa5e,163b7c695f7243779ef5e5f0fd844063,vehicle.car
4,9f0e8f997dc647bf9a0a20c679b1804d,RADAR_LEFT_FRONT,-0.048301,0.433370,0.896619,-0.149163,0.100862,0.100862,9f0e8f997dc647bf9a0a20c679b1804d,2c8422f4ac674f949fa1124dcaa1fa5e,163b7c695f7243779ef5e5f0fd844063,vehicle.car


In [13]:
radar_validation_df.to_csv(
    RESULT_DIR / "radar_validation_df.csv",
    index=False,
)

In [14]:
interval_keys = [
    "start_annotation_token",
    "end_annotation_token",
    "instance_token",
    "category",
]

radar_validation_interval_df = (
    radar_validation_df
    .groupby(interval_keys, as_index=False)
    .agg(
        radar_point_count=("radial_abs_error_mps", "size"),
        radial_abs_error_median_mps=("radial_abs_error_mps", "median"),
        radial_signed_bias_mps=("radial_residual_mps", "median"),
    )
)

radar_validation_interval_df.head()

,start_annotation_token,end_annotation_token,instance_token,category,radar_point_count,radial_abs_error_median_mps,radial_signed_bias_mps
0,00047c606afd439a9fd2e1977ffffaa5,c98ea9688f024269bbe3cde8d389d8ad,9c2474ba21d64b7284a302977fee4c3f,vehicle.car,3,0.180310,-0.180310
1,000e16e4591b4afdb8382f6ad0a60993,fddb24a95f7d448d9930c48225341181,82728f87d486472aa01fcb7871cd3252,vehicle.car,2,0.126338,-0.126338
2,00172d9e98194cfa8e229c6313b8459f,1452390c564046eb8334d37e1b8dcc12,c9ef7470245c4da8b8706a1da6dba80f,vehicle.truck,27,0.239670,0.236462
3,0028bcf2932f4b468d7990aa35969dee,5ce83f9fbe6a4b0da95a621c91b30331,0c948cb49ba04e51b79c4c390aa347b5,vehicle.car,3,0.121622,0.121622
4,002fba5e146b479c879e9e51fe402c9b,366320fecf4d46e38cd37043a085c7fa,eb60beb1aaca4394a4c5a75a1924907b,vehicle.car,10,0.261801,0.261801


In [15]:
errors = radar_validation_interval_df[
    "radial_abs_error_median_mps"
]

radar_native_summary = pd.Series({
    "intervals": len(radar_validation_interval_df),
    "radar_points": len(radar_validation_df),
    "median_abs_radial_error_mps": errors.median(),
    "radial_error_iqr_mps": (
        errors.quantile(0.75) - errors.quantile(0.25)
    ),
    "radial_error_p90_mps": errors.quantile(0.90),
    "median_signed_bias_mps": (
        radar_validation_interval_df[
            "radial_signed_bias_mps"
        ].median()
    ),
})

radar_native_summary

intervals                       9280.000000
radar_points                   50823.000000
median_abs_radial_error_mps        0.181880
radial_error_iqr_mps               0.310478
radial_error_p90_mps               0.917635
median_signed_bias_mps             0.003649
dtype: float64

In [16]:
reference_velocity_df = vehicle_reference_df[
    [
        "start_annotation_token",
        "end_annotation_token",
        "speed_2d_mps",
    ]
]

radar_check_df = radar_velocity_df.merge(
    reference_velocity_df,
    on=["start_annotation_token", "end_annotation_token"],
    how="left",
    validate="one_to_one",
)

radar_check_df["speed_abs_error_mps"] = (
    radar_check_df["radar_speed_2d_mps"]
    - radar_check_df["speed_2d_mps"]
).abs()

stable_df = radar_check_df[
    radar_check_df["is_stable_2d"]
].copy()

In [17]:
radar_2d_summary = pd.Series({
    "stable_intervals": len(stable_df),
    "median_2d_speed_error_mps": (
        stable_df["speed_abs_error_mps"].median()
    ),
    "p90_2d_speed_error_mps": (
        stable_df["speed_abs_error_mps"].quantile(0.90)
    ),
})

radar_2d_summary

stable_intervals             3016.000000
median_2d_speed_error_mps       0.276923
p90_2d_speed_error_mps         10.674228
dtype: float64

In [18]:
comparison_summary = pd.DataFrame({
    "Metric": [
        "Median error (m/s)",
        "P90 error (m/s)",
    ],
    "Native RADAR radial": [
        radar_native_summary["median_abs_radial_error_mps"],
        radar_native_summary["radial_error_p90_mps"],
    ],
    "Reconstructed RADAR 2D": [
        radar_2d_summary["median_2d_speed_error_mps"],
        radar_2d_summary["p90_2d_speed_error_mps"],
    ],
})

comparison_summary

,Metric,Native RADAR radial,Reconstructed RADAR 2D
0,Median error (m/s),0.181880,0.276923
1,P90 error (m/s),0.917635,10.674228


In [19]:
stable_keys = stable_df[
    [
        "start_annotation_token",
        "end_annotation_token",
    ]
].drop_duplicates()

stable_native_df = radar_validation_interval_df.merge(
    stable_keys,
    on=[
        "start_annotation_token",
        "end_annotation_token",
    ],
    how="inner",
)

stable_native_errors = stable_native_df[
    "radial_abs_error_median_mps"
]

stable_native_summary = pd.Series({
    "intervals": len(stable_native_df),
    "median_abs_radial_error_mps": stable_native_errors.median(),
    "radial_error_iqr_mps": (
        stable_native_errors.quantile(0.75)
        - stable_native_errors.quantile(0.25)
    ),
    "radial_error_p90_mps": stable_native_errors.quantile(0.90),
    "median_signed_bias_mps": stable_native_df[
        "radial_signed_bias_mps"
    ].median(),
})

stable_native_summary

intervals                      3016.000000
median_abs_radial_error_mps       0.205643
radial_error_iqr_mps              0.368091
radial_error_p90_mps              1.353229
median_signed_bias_mps            0.021187
dtype: float64

In [20]:
comparison_summary = pd.DataFrame({
    "Metric": [
        "Intervals",
        "Median error (m/s)",
        "P90 error (m/s)",
    ],
    "Native RADAR radial": [
        len(stable_native_df),
        stable_native_summary["median_abs_radial_error_mps"],
        stable_native_summary["radial_error_p90_mps"],
    ],
    "Reconstructed RADAR 2D": [
        len(stable_df),
        radar_2d_summary["median_2d_speed_error_mps"],
        radar_2d_summary["p90_2d_speed_error_mps"],
    ],
})

comparison_summary

,Metric,Native RADAR radial,Reconstructed RADAR 2D
0,Intervals,3016.000000,3016.000000
1,Median error (m/s),0.205643,0.276923
2,P90 error (m/s),1.353229,10.674228
